# Self-Discover: Composing Task-Specific Reasoning Structures

> **What you'll learn:** How to build a **Self-Discover** agent that first *discovers its own reasoning structure* for a novel task type, then uses that structure to solve a specific instance of the task.

Most agent patterns either answer a task directly, or answer-then-critique-and-revise (Reflection/Reflexion). **Self-Discover** takes a different approach: before attempting the task at all, the agent composes a **custom, step-by-step reasoning plan** tailored to the *type* of problem it's facing, and only then applies that plan to the specific instance.

The pattern (from Zhou et al., "SELF-DISCOVER: Large Language Models Self-Compose Reasoning Structures", and popularized in agentic-architecture catalogs such as `FareedKhan-dev/all-agentic-architectures`) breaks this into **four distinct stages**:

1. **SELECT** — From a fixed catalog of generic reasoning modules (e.g. "Critical Thinking", "Break down into sub-problems"), pick the subset relevant to this task.
2. **ADAPT** — Rephrase each selected generic module into a task-specific instruction.
3. **IMPLEMENT** — Compose the adapted instructions into a concrete, ordered, structured reasoning plan — the *reusable* "discovered structure" for this **task type**, not yet an answer to any specific instance.
4. **SOLVE** — Follow the discovered structure to actually solve the specific task instance and produce the final answer.

The key insight: stages 1–3 only look at the *shape* of the task (its type), while stage 4 is the only stage that touches the *specific* instance. In principle, the same discovered structure from stages 1–3 could be reused to solve many different instances of the same task type.

![Self-Discover](attachment:self-discover-diagram.png)

We'll demonstrate this on a **multi-constraint logic puzzle** — a task type where a model that just "answers directly" tends to drop or misapply one of the constraints, but a model that first discovers and follows an explicit reasoning structure (decomposition, case analysis, constraint propagation, verification) is much more likely to get it right.


### Setting up the Environment

We use the repo's unified LLM factory (`helpers.get_llm`) so this notebook works the same way across platforms (Groq on Windows, Databricks on macOS). We also use Pydantic models with `llm.with_structured_output(...)` for every stage so each stage's output is a clean, typed object we can inspect and pass to the next stage.

Set up your environment variables in a `.env` file at the project root as described in the main `CLAUDE.md` (e.g. `GROQ_API_KEY`, `OPENAI_API_KEY`, or Databricks credentials, depending on platform).

In [ ]:
# =============================================================================
# SETUP: Import dependencies and initialize the LLM
# =============================================================================

from typing import List

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# Import the unified LLM factory
from helpers import get_llm

# get_llm() auto-selects the best provider for your platform
llm = get_llm()

print("LLM initialized. Ready to run the Self-Discover pipeline.")

### The Reasoning Module Catalog

Self-Discover starts from a **fixed, generic catalog** of reasoning modules — short descriptions of ways of thinking that are useful across many kinds of problems. The agent never invents new modules; it only *selects* and *adapts* from this catalog. This is what makes the discovered structure reusable and inspectable, instead of an opaque chain-of-thought.

Below is a catalog of 10 generic reasoning modules, loosely modeled on the module list used in the original Self-Discover paper.

In [ ]:
# =============================================================================
# Purpose : Define the fixed catalog of generic reasoning modules
# Input   : None
# Output  : `REASONING_MODULES` — list of short generic module descriptions
# =============================================================================

REASONING_MODULES = [
    "Critical Thinking: Analyze the problem from multiple perspectives, question assumptions, "
    "and evaluate the evidence or reasoning at each step rather than accepting the first plausible answer.",

    "Decomposition: Break the problem down into smaller, more manageable sub-problems that can be "
    "addressed somewhat independently before combining their results.",

    "Use Analogies: Map the problem onto a similar, more familiar problem or scenario, and borrow "
    "the solution strategy that worked there.",

    "Step-by-Step Chronological Reasoning: Process the problem's information in a strict, ordered "
    "sequence, tracking how the state of the world evolves after each new piece of information.",

    "Consider Trade-offs: Weigh the pros and cons of multiple candidate solutions or interpretations "
    "before committing to one.",

    "Constraint Propagation: Identify the hardest, most restrictive constraints first, use them to "
    "eliminate impossible options immediately, and iteratively narrow the remaining solution space.",

    "Case Analysis: Enumerate the distinct possible cases or branches explicitly, test each one "
    "against every constraint, and discard any case that produces a contradiction.",

    "Working Backwards: Start from the desired end goal or the shape of a valid final answer, and "
    "reason backward about what conditions must have held earlier.",

    "Simplification: Restate the problem in the simplest possible terms, stripping away irrelevant "
    "details and flavor text, before attempting to solve it.",

    "Verification / Self-Check: After proposing an answer, re-check every stated constraint one by "
    "one against the proposed solution to confirm that none of them is violated.",
]

print(f"Catalog loaded with {len(REASONING_MODULES)} generic reasoning modules.")
for i, m in enumerate(REASONING_MODULES, 1):
    print(f"{i}. {m.split(':')[0]}")

### The Task

We'll use a small but genuinely tricky **multi-constraint scheduling puzzle**. It's small enough to verify by hand, but it has a conditional clue (an "if... then..." constraint) and an ordering clue — the kind of thing a model answering "off the top of its head" commonly gets wrong by dropping one constraint or mis-applying the conditional.

> Three analysts — **Priya**, **Quinn**, and **Rae** — must each be assigned to exactly one of three interview time slots: **9:00 AM**, **11:00 AM**, and **2:00 PM** (no two analysts share a slot).
>
> 1. Priya is **not** in the 9:00 AM slot.
> 2. Quinn's slot is **earlier** in the day than Rae's slot.
> 3. **If** Priya is in the 2:00 PM slot, **then** Quinn is in the 9:00 AM slot.
> 4. Rae is **not** in the 11:00 AM slot.
>
> Question: Which slot does each analyst have?

(For reference, the unique solution is **Quinn = 9:00 AM, Priya = 11:00 AM, Rae = 2:00 PM** — we'll use this to check the agent's final answer later, but we won't reveal it to the model.)

In [ ]:
# =============================================================================
# Purpose : Define the concrete task instance to solve
# Input   : None
# Output  : `TASK_TYPE_DESCRIPTION`, `TASK_INSTANCE` — the general task family and the specific puzzle
# =============================================================================

TASK_TYPE_DESCRIPTION = (
    "A multi-constraint logic/scheduling puzzle: several entities must each be assigned to exactly "
    "one slot/category out of a small fixed set, subject to a handful of constraints that include "
    "negative constraints, relative-ordering constraints, and conditional (if-then) constraints. "
    "Exactly one assignment satisfies all constraints simultaneously."
)

TASK_INSTANCE = (
    "Three analysts -- Priya, Quinn, and Rae -- must each be assigned to exactly one of three "
    "interview time slots: 9:00 AM, 11:00 AM, and 2:00 PM (no two analysts share a slot).\n\n"
    "Clues:\n"
    "1. Priya is not in the 9:00 AM slot.\n"
    "2. Quinn's slot is earlier in the day than Rae's slot.\n"
    "3. If Priya is in the 2:00 PM slot, then Quinn is in the 9:00 AM slot.\n"
    "4. Rae is not in the 11:00 AM slot.\n\n"
    "Question: Which slot does each analyst have?"
)

print("=== TASK TYPE ===")
print(TASK_TYPE_DESCRIPTION)
print("\n=== TASK INSTANCE ===")
print(TASK_INSTANCE)

### Baseline: Answering Directly (No Discovered Structure)

Before running Self-Discover, let's see what happens when we just ask the LLM to solve the puzzle directly, with no imposed structure. This gives us something to compare against later — direct-answer approaches to constraint puzzles like this one frequently drop a constraint (very often the conditional clue 3) or apply the ordering clue loosely.

In [ ]:
# =============================================================================
# Purpose : Establish a "just answer directly" baseline for comparison
# Input   : TASK_INSTANCE
# Output  : `baseline_answer` — the model's unstructured direct answer
# =============================================================================

baseline_prompt = ChatPromptTemplate.from_messages([
    ("system", "Solve the puzzle. Give a short, direct final answer only."),
    ("human", "{task}"),
])

baseline_chain = baseline_prompt | llm
baseline_answer = baseline_chain.invoke({"task": TASK_INSTANCE})

print("=== BASELINE (DIRECT ANSWER) ===")
print(baseline_answer.content)

**Discussion of the output:** Direct answers to this puzzle are inconsistent across models and temperatures — sometimes the ordering clue (2) is respected but the conditional clue (3) is silently ignored, or vice versa, because nothing forces the model to check *every* clue against the *final* proposed assignment. This is exactly the failure mode Self-Discover's explicit, verifiable structure is meant to prevent: instead of hoping the model juggles all constraints in one shot, we make it build and then follow a checklist.

Now let's run the four Self-Discover stages.

### Stage 1 — SELECT

In the **SELECT** stage, the LLM looks only at the **task type** (not the specific puzzle instance's answer) and the fixed module catalog, and chooses which modules are actually relevant. We use `llm.with_structured_output(ModuleSelection)` so the result is a clean, typed list we can hand to the next stage — no parsing free text.

In [ ]:
# =============================================================================
# Purpose : SELECT stage - choose relevant reasoning modules for this task type
# Input   : REASONING_MODULES, TASK_TYPE_DESCRIPTION
# Output  : `selection` — a ModuleSelection Pydantic object
# =============================================================================

class ModuleSelection(BaseModel):
    """The subset of the reasoning-module catalog relevant to this task type."""
    selected_modules: List[str] = Field(
        description="The exact module names (the text before the colon, e.g. 'Decomposition') "
        "copied verbatim from the provided catalog, for every module that is relevant to this task type."
    )
    rationale: str = Field(
        description="A brief explanation of why these modules were chosen and the others were not."
    )

select_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are the SELECT stage of a Self-Discover reasoning pipeline. You are given a catalog of "
        "generic reasoning modules and a description of a TASK TYPE (not a specific instance). "
        "Choose only the modules from the catalog that would genuinely help an agent reason through "
        "this type of task. Do not select modules that are irrelevant just to fill space, and do not "
        "invent new modules -- only choose from the catalog provided.",
    ),
    (
        "human",
        "Reasoning module catalog:\n{catalog}\n\nTask type:\n{task_type}\n\n"
        "Select the modules relevant to this task type.",
    ),
])

selector = select_prompt | llm.with_structured_output(ModuleSelection)

catalog_text = "\n".join(f"- {m}" for m in REASONING_MODULES)
selection = selector.invoke({"catalog": catalog_text, "task_type": TASK_TYPE_DESCRIPTION})

print("=== STAGE 1: SELECT ===")
print("Selected modules:")
for m in selection.selected_modules:
    print(f"  - {m}")
print(f"\nRationale: {selection.rationale}")

### Stage 2 — ADAPT

The generic modules selected above are still phrased abstractly (e.g. "Break the problem into sub-problems"). In the **ADAPT** stage, the LLM rephrases each selected module into an instruction specific to *this task type* — still without touching the specific puzzle instance's answer. For example, "Decomposition" might become "Treat each clue as a separate sub-problem and note what it rules out."

In [ ]:
# =============================================================================
# Purpose : ADAPT stage - rephrase each selected module into a task-specific instruction
# Input   : selection.selected_modules, TASK_TYPE_DESCRIPTION
# Output  : `adaptation` — an AdaptedModules Pydantic object
# =============================================================================

class AdaptedModules(BaseModel):
    """Task-specific rephrasing of each selected generic reasoning module."""
    adapted_instructions: List[str] = Field(
        description="One task-specific instruction per selected module, in the same order as the "
        "selected modules, rephrasing each generic module in language specific to this task type."
    )

adapt_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are the ADAPT stage of a Self-Discover reasoning pipeline. You are given a TASK TYPE "
        "and a list of generic reasoning modules already selected as relevant. Rephrase each generic "
        "module into a concrete instruction specific to this task type -- still generic to the task "
        "TYPE, not to any one specific instance of it.",
    ),
    (
        "human",
        "Task type:\n{task_type}\n\nSelected generic modules:\n{modules}\n\n"
        "Adapt each module into a task-specific instruction, preserving order.",
    ),
])

adapter = adapt_prompt | llm.with_structured_output(AdaptedModules)

modules_text = "\n".join(f"- {m}" for m in selection.selected_modules)
adaptation = adapter.invoke({"task_type": TASK_TYPE_DESCRIPTION, "modules": modules_text})

print("=== STAGE 2: ADAPT ===")
for i, instr in enumerate(adaptation.adapted_instructions, 1):
    print(f"{i}. {instr}")

### Stage 3 — IMPLEMENT

Now the LLM composes the adapted instructions into a single, **ordered, structured reasoning plan** — the actual "discovered structure" for this task type. This is still *not* an answer to the puzzle: it's a reusable checklist/algorithm that could in principle be applied to any instance of this task type (a different set of names, slots, and clues). We represent it as a Pydantic `ReasoningStructure` — an ordered list of numbered steps — so it can be printed, inspected, and (in a more advanced system) cached and reused.

In [ ]:
# =============================================================================
# Purpose : IMPLEMENT stage - compose adapted instructions into an ordered reasoning structure
# Input   : adaptation.adapted_instructions, TASK_TYPE_DESCRIPTION
# Output  : `reasoning_structure` — a ReasoningStructure Pydantic object (the discovered structure)
# =============================================================================

class ReasoningStep(BaseModel):
    step_number: int = Field(description="1-indexed position of this step in the reasoning plan.")
    instruction: str = Field(description="A single, concrete, actionable instruction for this step.")

class ReasoningStructure(BaseModel):
    """The reusable, discovered step-by-step reasoning plan for this task TYPE."""
    task_type: str = Field(description="A short label naming the general category of task this plan is for.")
    steps: List[ReasoningStep] = Field(description="The ordered sequence of steps making up the plan.")

implement_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are the IMPLEMENT stage of a Self-Discover reasoning pipeline. You are given a TASK "
        "TYPE and a set of task-specific adapted instructions. Compose them into ONE ordered, "
        "concrete, step-by-step reasoning plan (an algorithm) that an agent could follow to solve "
        "ANY specific instance of this task type. Do not solve any particular instance here -- "
        "produce only the reusable plan/structure, with each step phrased as an imperative "
        "instruction (e.g. 'List all entities and all possible slots.').",
    ),
    (
        "human",
        "Task type:\n{task_type}\n\nAdapted, task-specific instructions to compose:\n{instructions}\n\n"
        "Produce the ordered reasoning structure.",
    ),
])

implementer = implement_prompt | llm.with_structured_output(ReasoningStructure)

instructions_text = "\n".join(f"- {i}" for i in adaptation.adapted_instructions)
reasoning_structure = implementer.invoke({
    "task_type": TASK_TYPE_DESCRIPTION,
    "instructions": instructions_text,
})

print("=== STAGE 3: IMPLEMENT -- DISCOVERED REASONING STRUCTURE ===")
print(f"Task type: {reasoning_structure.task_type}\n")
for step in reasoning_structure.steps:
    print(f"Step {step.step_number}: {step.instruction}")

**Discussion of the output:** Notice that the printed structure above is a self-contained checklist that says nothing about Priya, Quinn, Rae, or any specific time slot — it's generic to "multi-constraint assignment puzzles." This is the defining feature of Self-Discover: **stages 1–3 never look at the answer**, only at the shape of the problem. That's what makes the discovered structure reusable across many puzzle instances of the same type, and it's also what makes it inspectable/debuggable before we ever spend effort solving a specific case.

### Stage 4 — SOLVE

Finally, we hand the LLM **both** the discovered structure **and** the specific puzzle instance, and ask it to follow the structure step by step to produce the final answer. This is the only stage that touches `TASK_INSTANCE`.

In [ ]:
# =============================================================================
# Purpose : SOLVE stage - follow the discovered structure to solve the specific task instance
# Input   : reasoning_structure, TASK_INSTANCE
# Output  : `solution` — a FinalAnswer Pydantic object with a worked trace and final answer
# =============================================================================

class FinalAnswer(BaseModel):
    reasoning_trace: str = Field(
        description="The worked reasoning, following the discovered structure step by step, "
        "including any case analysis and the final verification pass against every clue."
    )
    final_answer: str = Field(
        description="The concrete final answer: the slot assigned to each analyst."
    )

structure_text = "\n".join(f"Step {s.step_number}: {s.instruction}" for s in reasoning_structure.steps)

solve_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are the SOLVE stage of a Self-Discover reasoning pipeline. You are given a discovered "
        "reasoning structure (a numbered checklist) and a specific task instance. Follow the "
        "structure step by step -- do not skip steps, especially any verification step -- and use it "
        "to solve the specific instance. Show your work in `reasoning_trace`, and give the concrete "
        "final assignment in `final_answer`.",
    ),
    (
        "human",
        "Discovered reasoning structure:\n{structure}\n\nTask instance:\n{instance}",
    ),
])

solver = solve_prompt | llm.with_structured_output(FinalAnswer)
solution = solver.invoke({"structure": structure_text, "instance": TASK_INSTANCE})

print("=== STAGE 4: SOLVE ===")
print("--- Reasoning trace ---")
print(solution.reasoning_trace)
print("\n--- Final answer ---")
print(solution.final_answer)

### Comparing the Two Approaches

The correct assignment is **Quinn = 9:00 AM, Priya = 11:00 AM, Rae = 2:00 PM** (you can verify this by checking all four clues against it — no other assignment satisfies all of them simultaneously).

Compare `baseline_answer.content` (Stage-less, direct answer) against `solution.final_answer` (produced by following the discovered structure). The structured path is far more likely to have explicitly worked through the conditional clue (3) and to have run a final verification pass against all four clues — because the discovered structure *made those steps mandatory* rather than optional. The baseline has no such guarantee: it succeeds only if the model happens to juggle all constraints correctly in a single pass.

### Key Takeaways

- **Self-Discover separates "how to think about this kind of problem" from "solving this specific problem."** The SELECT → ADAPT → IMPLEMENT stages only ever look at the task *type*; only the final SOLVE stage touches the specific instance.
- **The discovered structure is reusable and inspectable.** Because it's produced as a typed `ReasoningStructure` (an ordered list of steps), it can be printed, logged, cached, and — in a more advanced system — reused across many instances of the same task type without re-running SELECT/ADAPT/IMPLEMENT each time.
- **Structured output at every stage (`llm.with_structured_output(...)`) keeps the pipeline reliable.** Each stage produces a typed Pydantic object instead of free text, so downstream stages can consume it programmatically rather than re-parsing prose.
- **The value shows up most clearly on tasks with easy-to-drop constraints** (conditionals, orderings, negative constraints) — exactly where "just answer directly" tends to silently skip a check that an explicit, mandatory structure forces the model to perform.
- **Where this fits in the roadmap:** Self-Discover belongs to **Layer B3** of the agentic-pattern catalog, alongside **Reflection**, **Reflexion**, and **Chain-of-Verification (CoVe)** — all four are "self-improving reasoning" patterns that spend extra inference-time compute to catch mistakes a single direct pass would make, but each does so differently: Reflection/Reflexion critique and revise an *already-produced* output, CoVe verifies an output via follow-up questions, while Self-Discover front-loads the improvement by discovering the *right reasoning process* before ever attempting an answer.